# FreeCalliope Colab

Run a free CalliopeLabs-style faceless animation video generator in Google Colab.

Use GPU runtime for speed: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
# Setup
REPO_URL = "https://github.com/developer5432112345-cmyk/freecalliope.git"
!test -d freecalliope || git clone $REPO_URL freecalliope
%cd freecalliope
!pip install -q -r requirements-colab.txt


In [ ]:
# Video Generator UI
import ipywidgets as widgets
from IPython.display import display, clear_output
from freecalliope.colab_pipeline import run_pipeline

result = None

topic = widgets.Textarea(
    value="What did ancient humans actually do all day?",
    description="Topic",
    layout=widgets.Layout(width="100%", height="90px"),
)
style = widgets.Dropdown(
    options=["auto", "ink_explainer", "finance", "stickman_story", "documentary", "history", "scary_story", "general"],
    value="auto",
    description="Style",
)
platform = widgets.Dropdown(options=["youtube", "shorts", "tiktok", "reels"], value="youtube", description="Platform")
generation_mode = widgets.Dropdown(options=["flow_fast", "classic_fast"], value="flow_fast", description="Mode")
minutes = widgets.FloatSlider(value=3, min=0.5, max=12, step=0.5, description="Minutes", readout_format=".1f")
scene_count = widgets.IntSlider(value=12, min=4, max=48, step=1, description="Scenes")

image_model = widgets.Dropdown(
    options=["stabilityai/sdxl-turbo", "stabilityai/sd-turbo"],
    value="stabilityai/sdxl-turbo",
    description="Model",
    layout=widgets.Layout(width="70%"),
)
steps = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Steps")

voice_provider = widgets.Dropdown(options=["edge", "gtts", "omnivoice"], value="edge", description="Voice")
edge_voice = widgets.Dropdown(
    options=["en-US-AriaNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural", "en-GB-SoniaNeural"],
    value="en-US-AriaNeural",
    description="Edge",
)
omnivoice_audio_path = widgets.Text(value="", placeholder="/content/omnivoice_output.wav", description="OmniVoice", layout=widgets.Layout(width="100%"))
gemini_api_key = widgets.Password(value="", placeholder="Optional Google AI Studio key", description="Gemini", layout=widgets.Layout(width="100%"))

generate_button = widgets.Button(description="Generate Video", button_style="success", icon="video-camera")
output = widgets.Output()

def selected_size():
    if platform.value in ["shorts", "tiktok", "reels"]:
        return 576, 1024
    return 1024, 576

def on_generate(_):
    global result
    with output:
        clear_output()
        width, height = selected_size()
        print("Generating video. This can take several minutes...")
        result = run_pipeline(
            topic=topic.value.strip(),
            style=style.value,
            platform=platform.value,
            generation_mode=generation_mode.value,
            minutes=minutes.value,
            scene_count=scene_count.value,
            gemini_key=gemini_api_key.value or None,
            image_model=image_model.value,
            width=width,
            height=height,
            steps=steps.value,
            guidance_scale=0.0,
            voice_provider=voice_provider.value,
            edge_voice=edge_voice.value,
            omnivoice_audio_path=omnivoice_audio_path.value or None,
        )
        print("Done.")
        print("Video:", result.get("video_path"))
        print("Zip:", result.get("zip_path"))

generate_button.on_click(on_generate)

display(widgets.VBox([
    widgets.HTML("<h3>FreeCalliope Video Generator</h3>"),
    topic,
    widgets.HBox([style, platform, generation_mode]),
    widgets.HBox([minutes, scene_count, steps]),
    image_model,
    widgets.HBox([voice_provider, edge_voice]),
    omnivoice_audio_path,
    gemini_api_key,
    generate_button,
    output,
]))


In [ ]:
# Download the generated MP4 and project zip
from google.colab import files
if not result:
    raise RuntimeError("Generate a video first using the UI cell above.")
files.download(result["video_path"])
files.download(result["zip_path"])
